In [1]:
from stanza.utils.conll import CoNLL
import pandas as pd
import os
import Utils
from collections import Counter
import json

# Identifying Conditionals in corrected texts

We use the corrected texts because they contain the target hypotheses and we are not doing any accuracy-related research.

In [2]:
cor_path = f"../parsed_documents/corrected/"
cor_files = sorted(os.listdir(cor_path))

In [ ]:
def get_if(sentence):
    for w in sentence.all_words:
        if w.lemma.lower() == "if":
            return w
    return None    

        

def get_conditionals(doc, student_id, group, topic, writing_id):
    """
    Returns a list of dictionaries with information about the conditional clauses found in the Stanza doc.
    The dictionaries include information about the conditional
    """
    dict_list = []
    
    # char_alignment = alignment_pickle[student_id]["c2o_char_align"]
    # Each instance of a conditional corresponds to a dict in the return dict_list
    for cor_sent in doc.sentences:
        found_if = get_if(cor_sent)
        
        # No "if" found in the sentence
        if not found_if:
            continue
        
        
        if_clause_head = cor_sent.all_words[found_if.head - 1]
        if_clause_aux = [c for c in Utils.get_children_ud(cor_sent, if_clause_head) if c.deprel == "aux"]
        main_clause_head = None
        main_clause_modal = None
        aux_main_clause = None
        
        
        # Relationship between the head of the sentence and the head of the If-clause: If ccomp, if is a complementizer, not a conditional marker
        if if_clause_head.deprel and "ccomp" in if_clause_head.deprel:
            # print(f"Non-conditional if: {cor_sent.text}")
            continue
        
        for word in cor_sent.all_words:

            if (word.head == 0):
                # Ideally, a verb will be the head, but it may also be a noun or adjective (attributive sentences)
                if "VB" in word.xpos:
                    main_clause_head = word
                
                else:
                    main_clause_head = [c for c in Utils.get_children_ud(cor_sent, word) if "VB" in c.xpos][0]
                    
                aux_main_clause =  [c for c in Utils.get_children_ud(cor_sent, main_clause_head) if c.deprel == "aux"]
            
            else:
                # modals allowed in apodosis of CII
                is_cii_aux = word.lemma.lower() in {"could", "would", "might", "may"}
                # whether the word is inside the if-clause
                is_in_if_clause = word.head == found_if.head
                
                if (is_cii_aux and not is_in_if_clause and main_clause_modal is None):
                    main_clause_modal = word
        

        if_head_past = if_clause_head.feats and "Tense=Past" in if_clause_head.feats
        
        
        
        # Add identified sentence to df 
        cond_dict = {
                    "topic": topic,
                    "student_id": student_id,
                    "writing_id": writing_id,
                    "group": group,
                    "is_cii": if_head_past and main_clause_modal is not None and not if_clause_aux,
                    "cor_sent" : cor_sent.text,
                    "cor_if_head": if_clause_head.text,
                    "cor_if_head_lemma": if_clause_head.lemma,
                    "cor_main_head": main_clause_head.text,
                    "cor_main_aux": aux_main_clause[0].text if aux_main_clause else None,
                }
               
        dict_list.append(cond_dict)
                        
        
    return dict_list




## Running the code on our data

In [4]:
output_path = "../Python output files/"
os.makedirs(output_path, exist_ok=True)

topic_word_count = Counter()

def update_topic_words_counter(doc, topic):
    n_words = sum(len(sent.all_words) for sent in doc.sentences)
    topic_word_count[topic] += n_words
            

In [58]:
output_file = output_path + "/02_all_conditionals.csv"
log_file = f"../{output_path}/unprocessed_filenames.txt"



# Load existing parsed data
if os.path.exists(output_file):
    print("Output file exists: resuming run")
    found_conds = pd.read_csv(output_file)
    cond_list = found_conds.to_dict("records")
    processed_ids = set(found_conds["writing_id"].astype(str))
else:
    cond_list = []
    processed_ids = set()



# Load unprocessed log safely
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        unprocessed_files = f.read().splitlines()
else:
    unprocessed_files = []



# Parse
for i, filename in enumerate(cor_files):

    if i % 1000 == 0:
        print(f"File {i} of {len(cor_files)}")

    filename_split = filename.split("_")

    student_id = filename_split[-1][:-6]
    group = filename_split[0]
    topic = filename_split[1]
    writing_id = filename_split[2]  
    
    path2file = os.path.join(cor_path, filename)

    # Skip if already processed
    if writing_id in processed_ids:
        # print("File already processed")
        continue

    # Skip missing files
    if not os.path.isfile(path2file):
        continue

    # Parse document
    doc = None
    try:
        doc = CoNLL.conll2doc(path2file)
        cond_clauses_dicts = get_conditionals(
            doc, student_id, group, topic, writing_id
        )
        
    except Exception:
        unprocessed_files.append(filename)
        continue
    
    # Count the number of words that this file has added to its "topic"
    update_topic_words_counter(doc, topic)
    
    # Append conditional results
    for d in cond_clauses_dicts:
        cond_list.append(d)

    # Mark as processed
    processed_ids.add(writing_id)

    # Periodic save
    if i % 3000 == 0 and len(cond_list) > 0:
        print("Saving checkpoint...")
        pd.DataFrame(cond_list).to_csv(output_file, index=False)
        print("Saved.")    


# Save final results
found_conds = pd.DataFrame(cond_list)
found_conds.to_csv(output_file, index=False)



cii = found_conds[found_conds["is_cii"] == True]
cii.to_csv(
    output_path + "/02_conditionals_ii.csv",
    index=False
)


with open(f"../{output_path}/topic_word_count.json", "w") as count_file:
    json.dump(dict(topic_word_count), count_file)



# Save unprocessed files
existing_unprocessed = set()

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        existing_unprocessed = set(f.read().splitlines())

new_unprocessed = [f for f in unprocessed_files if f not in existing_unprocessed]

with open(log_file, "a") as f:
    for fname in new_unprocessed:
        f.write(fname + "\n")

File 0 of 33496
Saving checkpoint...
Saved.
File 1000 of 33496
File 2000 of 33496
File 3000 of 33496
Saving checkpoint...
Saved.
File 4000 of 33496
File 5000 of 33496
File 6000 of 33496
Saving checkpoint...
Saved.
File 7000 of 33496
File 8000 of 33496
File 9000 of 33496
Saving checkpoint...
Saved.
File 10000 of 33496
File 11000 of 33496
File 12000 of 33496
Saving checkpoint...
Saved.
File 13000 of 33496
File 14000 of 33496
File 15000 of 33496
Saving checkpoint...
Saved.
File 16000 of 33496
File 17000 of 33496
File 18000 of 33496
Saving checkpoint...
Saved.
File 19000 of 33496
File 20000 of 33496
File 21000 of 33496
Saving checkpoint...
Saved.
File 22000 of 33496
File 23000 of 33496
File 24000 of 33496
Saving checkpoint...
Saved.
File 25000 of 33496
File 26000 of 33496
File 27000 of 33496
Saving checkpoint...
Saved.
File 28000 of 33496
File 29000 of 33496
File 30000 of 33496
Saving checkpoint...
Saved.
File 31000 of 33496
File 32000 of 33496
File 33000 of 33496
Saving checkpoint...
Save

In [59]:
len(cii)

585